# 黑马头条项目｜第07天：APP端文章搜索

> 学习目标：用 Elasticsearch 实现文章全文搜索、时间分页和标题高亮；通过 Kafka 同步新文章索引；用 MongoDB 保存、查看和删除用户搜索历史，并实现关键字联想词。

本篇根据百度网盘 `day07-app端文章搜索/讲义/app端文章搜索.md` 网页预览整理，衔接第04天文章审核与静态化、第05天定时审核、第06天 Kafka 消息通知。未读取配套源码、PPT或导入脚本，未启动 Elasticsearch、IK、MongoDB、Kafka、Nacos、MySQL、Docker或课程微服务，未执行环境安装、数据导入和接口测试。

示例沿用课程的 Elasticsearch 7.4.0 与 Java `RestHighLevelClient`；它们属于课程版本，实际新项目应按目标版本选择客户端。下文注明“补充”“修正”的逻辑用于解释边界，不代表原讲义已经实现。预览中的图片未能加载，本篇使用文字流程和代码说明，没有嵌入原图。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 今日业务与存储分工

APP搜索页面支持三种行为：输入关键词并搜索文章；查看和删除自己的搜索历史；输入部分关键词时显示联想词。文章搜索结果具有标题高亮，列表字段与首页接近，点击后可查看详情。

| 功能 | 数据来源 | 本课技术 | 主要接口 |
|---|---|---|---|
| 文章全文搜索 | 已发布文章的索引副本 | Elasticsearch | POST `/api/v1/article/search/search` |
| 新文章索引同步 | article服务生成静态页后发送事件 | Kafka → Elasticsearch | Topic `article.es.sync.topic` |
| 个人搜索历史 | 当前用户的搜索关键词 | MongoDB | POST `/api/v1/history/load`、`/del` |
| 联想词 | 预先维护的公共词库 | MongoDB regex | POST `/api/v1/associate/search` |

```text
MySQL文章数据 → 初次批量导入 → Elasticsearch app_info_article
article生成静态页 → Kafka增量事件 → 搜索服务更新文档
                                      ↑
APP搜索请求 → 搜索服务 → ES复合查询 → 列表 + h_title
                  └→ 登录用户首次查询：异步保存Mongo搜索历史

APP输入关键词 → Mongo联想词集合 → 候选词列表
APP打开历史页 → 当前用户的历史集合 → 最新10条
```

MySQL维护文章业务数据，ES提供搜索用的读模型。MongoDB维护搜索历史与公共词库。ES和MongoDB不是本项目文章主表的替代品。


## 2. Elasticsearch环境与IK中文分词

讲义使用镜像 `elasticsearch:7.4.0`，单节点模式 `discovery.type=single-node`，HTTP端口9200、节点通信端口9300。资料中的 `elasticsearch-analysis-ik-7.4.0.zip` 必须与ES版本匹配，解压到插件目录后重启才能按课程方式加载。

关键概念：

- **Index（索引）**：存放一类可搜索文档，课程名称是 `app_info_article`；
- **Document（文档）**：一篇文章在ES中的JSON数据；
- **Mapping（映射）**：规定字段类型、是否索引及分词方式；
- **Analyzer（分析器）**：将原文转换为检索词项，影响搜索匹配结果；
- **IK中文分词**：本课标题和正文指定 `ik_smart`。

验证IK可使用课程环境的分析接口，下面仅列出请求，不在本次执行：

```http
POST <ES_BASE_URL>/_analyze
Content-Type: application/json

{
  "analyzer": "ik_smart",
  "text": "黑马头条文章搜索"
}
```

ES分词器影响索引建立和查询分析。不能把全文搜索等同于对原始字符串执行 `LIKE '%关键词%'`。已有字段的类型和分词器通常不能简单原地修改，需要新建索引并重建数据。


## 3. app_info_article的映射设计

课程的索引创建请求为 `PUT <ES_BASE_URL>/app_info_article`，映射如下：

```json
{
  "mappings": {
    "properties": {
      "id":          { "type": "long" },
      "publishTime": { "type": "date" },
      "layout":      { "type": "integer" },
      "images":      { "type": "keyword", "index": false },
      "staticUrl":   { "type": "keyword", "index": false },
      "authorId":    { "type": "long" },
      "authorName":  { "type": "text" },
      "title":       { "type": "text", "analyzer": "ik_smart" },
      "content":     { "type": "text", "analyzer": "ik_smart" }
    }
  }
}
```

| 字段 | 用途 | 设计含义 |
|---|---|---|
| id | 文章ID | Long，与APP文章ID一致；写入时也用作ES `_id` |
| publishTime | 发布时间 | 日期类型，支持范围过滤和倒序排序 |
| title/content | 关键词检索 | text字段分词，匹配词项 |
| layout/images | 列表封面展示 | 图片地址无需全文检索 |
| staticUrl | 详情静态页 | 保留在返回数据中，无需搜索 |
| authorId/authorName | 作者展示 | 本课查询主要检索title和content |

`index:false`表示不为该字段建立通常的搜索索引，不是删除字段；数据仍可以从 `_source` 返回。文档 `_id` 和 `_source.id` 是不同位置，课程写入时让两者对应同一篇文章。

讲义还列出查看映射、查询文档、删除索引等管理请求。本笔记不执行这些请求。删除索引会删除搜索数据；修改映射或重建索引前应另行准备迁移和回滚方案。


## 4. 搜索文章VO与数据库初始导入

课程将搜索所需字段封装为 `SearchArticleVo`，增量同步时放入公共model工程：

```java
@Data
public class SearchArticleVo {
    private Long id;
    private String title;
    private Date publishTime;
    private Integer layout;
    private String images;
    private Long authorId;
    private String authorName;
    private String staticUrl;
    private String content;
}
```

初次搭建时导入测试模块中的 `es-init`，通过 `ApArticleMapper.loadArticleList()` 查询文章列表，并使用Bulk API写入。讲义未展示该Mapper查询SQL，因此这里不推断它已正确过滤下架、删除或未发布文章。

```java
List<SearchArticleVo> articles = apArticleMapper.loadArticleList();
BulkRequest bulk = new BulkRequest();
for (SearchArticleVo article : articles) {
    bulk.add(new IndexRequest("app_info_article")
            .id(article.getId().toString())
            .source(JSON.toJSONString(article), XContentType.JSON));
}
BulkResponse response = restHighLevelClient.bulk(
        bulk, RequestOptions.DEFAULT);
// 补充：逐项检查失败，不能仅认为请求没有抛异常就全部成功。
if (response.hasFailures()) {
    // 保存失败项、失败原因，按批次重试或补偿。
}
```

数据量大时分页读取、分批发送Bulk，按文档大小与集群承载能力设置批次，避免一次载入所有文章。使用固定 `_id` 便于重复导入覆盖同一文档，但仍要防止旧数据覆盖新版本。

讲义增量代码把文章正文JSON整体赋给 `content`。工程补充建议抽取正文中的纯文本，避免图片URL和结构字段影响分词与召回，并保持初始导入、增量同步的转换规则一致。


## 5. 搜索微服务与APP网关路由

新增 `heima-leadnews-search`，服务名 `leadnews-search`。课程依赖 `elasticsearch-rest-high-level-client`、`elasticsearch-rest-client`、`elasticsearch`，版本均为7.4.0；公共依赖管理应保持版本兼容。

搜索服务主要连接ES、MongoDB和Kafka。课程排除JDBC数据源自动配置，避免因公共依赖引入JDBC但未配置关系型数据源导致启动异常。

```yaml
spring:
  application:
    name: leadnews-search
  autoconfigure:
    exclude: org.springframework.boot.autoconfigure.jdbc.DataSourceAutoConfiguration
elasticsearch:
  host: ${ES_HOST}
  port: ${ES_HTTP_PORT}
```

`elasticsearch.host/port` 是课程的自定义配置项，要配合课程提供的客户端配置类构造 `RestHighLevelClient`；仅写YAML并不必然自动生成该Bean。所有地址使用占位符。

APP网关添加路由片段：

```yaml
- id: leadnews-search
  uri: lb://leadnews-search
  predicates:
    - Path=/search/**
  filters:
    - StripPrefix=1
```

例如外部请求 `/search/api/v1/article/search/search`，去掉第一段后进入搜索服务 `/api/v1/article/search/search`。历史和联想词接口也可以走该 `/search/**` 路由。课程联调要求启动article、user、search、APP网关和APP前端；本次没有启动这些服务。


## 6. 搜索接口与UserSearchDto

服务内部接口：`POST /api/v1/article/search/search`，响应为 `ResponseResult` 包装的文章列表。

```java
@Data
public class UserSearchDto {
    private String searchWords;  // 搜索关键词
    private int pageNum;         // 当前页
    private int pageSize;        // 每批数量
    private Date minBehotTime;   // 拉取更早文章的时间边界

    public int getFromIndex() {
        if (pageNum < 1) return 0;
        if (pageSize < 1) pageSize = 10;
        return pageSize * (pageNum - 1);
    }
}
```

讲义查询实际使用 `from(0)`，依靠 `publishTime < minBehotTime` 拉取更早的文章；它没有使用 `getFromIndex()` 的结果作为ES offset。`getFromIndex()==0` 在后续代码中用于判断是否记录搜索历史。

```java
@RestController
@RequestMapping("/api/v1/article/search")
public class ArticleSearchController {
    @Autowired
    private ArticleSearchService articleSearchService;

    @PostMapping("/search")
    public ResponseResult search(@RequestBody UserSearchDto dto)
            throws IOException {
        return articleSearchService.search(dto);
    }
}
```

入口至少校验：dto不为空、关键词trim后非空、关键词长度合理、pageSize在允许范围内、首次请求时间边界有默认值。不要依赖具有副作用的 `getFromIndex()` 来统一修正参数。


## 7. ES复合查询：关键词、时间、排序与数量

课程查询包含四个条件：

1. 关键词在 `title` 或 `content` 中匹配；
2. `publishTime` 小于本次时间边界；
3. 以发布时间倒序返回；
4. 取 `pageSize` 条。

```java
BoolQueryBuilder boolQuery = QueryBuilders.boolQuery();
boolQuery.must(QueryBuilders.queryStringQuery(dto.getSearchWords())
        .field("title")
        .field("content")
        .defaultOperator(Operator.OR));
boolQuery.filter(QueryBuilders.rangeQuery("publishTime")
        .lt(dto.getMinBehotTime().getTime()));

SearchSourceBuilder source = new SearchSourceBuilder()
        .query(boolQuery)
        .from(0)
        .size(dto.getPageSize())
        .sort("publishTime", SortOrder.DESC);

SearchRequest request = new SearchRequest("app_info_article")
        .source(source);
SearchResponse response = restHighLevelClient.search(
        request, RequestOptions.DEFAULT);
```

`must` 承载关键词匹配；`filter` 承载时间筛选，不要求该条件参与相关性打分。课程最终按发布时间排序，因此不是以 `_score` 为主的相关性排序。

`Operator.OR` 允许分词后的部分词项命中，并非要求所有词同时出现。若期望更严格匹配，需结合业务设计AND、短语查询或最小匹配比例，而不是直接改成字符串相等。

**query_string的边界：** 它支持查询语法，括号、冒号等用户输入可能影响解析或抛出异常。面向普通搜索框时，可补充使用 `multiMatchQuery(searchWords, "title", "content")` 或受限的simple_query_string，避免用户必须理解查询语法。此替换是笔记建议，课程实现使用query_string。


## 8. 时间分页与相同时间的边界

课程从时间边界之前取一批文章：

```text
首次：关键词 + 初始minBehotTime → 按时间倒序取10条
后续：关键词 + 上一批最小publishTime → 继续取更早10条
```

这种分页减小深分页offset的开销，但只用一个时间字段会漏掉同一时刻的记录。例：12:00:00有15篇文章，第一页返回其中10篇；下一页条件严格小于12:00:00，剩余5篇会被跳过。改成 `<=` 又可能反复返回上一批。

补充方案：定义唯一且稳定的排序 `(publishTime DESC, id DESC)`，游标携带两个值，用 `search_after` 继续读取；需要一致的搜索快照时可结合目标ES版本提供的PIT功能。不要在讲义原查询上同时叠加pageNum offset和时间游标，否则可能重复跳过数据。

### 讲义参数问题

- 直接调用 `dto.getMinBehotTime().getTime()`，为空时会抛异常，应设置首次边界或明确必填。
- 原 `getFromIndex()` 在pageNum<1时直接返回，未修复pageSize；如果size仍为0，搜索可能返回空结果。
- 联想词仅限制pageSize最大20，搜索接口本身也需要限定页大小。
- 搜索过程中更换关键词，前端应重置页码和时间游标。

时间分页得到的是按时间分批的搜索结果，不是固定总页数的传统分页。响应最好包含下一页游标及是否还有数据，但课程这里只返回列表。


## 9. 标题高亮与结果封装

课程只高亮title，使用自定义前后标签：

```java
HighlightBuilder highlighter = new HighlightBuilder()
        .field("title")
        .preTags("<font style='color: red; font-size: inherit;'>")
        .postTags("</font>");
source.highlighter(highlighter);
```

返回时将 `_source` 转为Map，并追加 `h_title`：有高亮则使用高亮片段，否则使用原始title。

```java
List<Map<String, Object>> result = new ArrayList<>();
for (SearchHit hit : response.getHits().getHits()) {
    Map<String, Object> article = JSON.parseObject(
            hit.getSourceAsString(), Map.class);
    HighlightField titleHighlight = hit.getHighlightFields().get("title");
    if (titleHighlight != null && titleHighlight.getFragments() != null
            && titleHighlight.getFragments().length > 0) {
        String highlightedTitle = Arrays.stream(titleHighlight.getFragments())
                .map(Text::string).collect(Collectors.joining());
        article.put("h_title", highlightedTitle);
    } else {
        article.put("h_title", article.get("title"));
    }
    result.add(article);
}
return ResponseResult.okResult(result);
```

这里补充检查title具体高亮字段是否存在；不能仅检查“高亮Map不为空”就直接访问title。标题没有关键词而正文命中时，title可能不会有高亮。

前端若将 `h_title` 作为HTML渲染，应对原始标题内容编码，并只允许规定的高亮标签。可使用ES高亮HTML编码配置及前端白名单处理；不能把任意用户提交的标题当作可信HTML。上面的font标签保留课程格式，产品样式可改为受控span。


## 10. 新文章增量索引：静态页之后发送Kafka消息

第04天article服务生成详情HTML、上传MinIO并回写 `static_url`。本日扩展该流程，在静态页地址准备好后收集搜索VO，并发布到 `article.es.sync.topic`。

```text
审核通过 → article三表保存 → Freemarker渲染
                          ↓
                     HTML上传MinIO
                          ↓
                 回写ap_article.static_url
                          ↓
              构造SearchArticleVo并发送Kafka
```

关键摘录：

```java
public static final String ARTICLE_ES_SYNC_TOPIC =
        "article.es.sync.topic";

private void createArticleESIndex(ApArticle article,
                                  String content, String path) {
    SearchArticleVo vo = new SearchArticleVo();
    BeanUtils.copyProperties(article, vo);
    vo.setContent(content);
    vo.setStaticUrl(path);
    kafkaTemplate.send(ArticleConstants.ARTICLE_ES_SYNC_TOPIC,
                       JSON.toJSONString(vo));
}
```

article服务需要生产者配置，search服务需要消费者配置：

```yaml
spring:
  kafka:
    bootstrap-servers: ${KAFKA_BOOTSTRAP_SERVERS}
    producer:
      retries: 10
      key-serializer: org.apache.kafka.common.serialization.StringSerializer
      value-serializer: org.apache.kafka.common.serialization.StringSerializer
    consumer:
      group-id: ${spring.application.name}
      key-deserializer: org.apache.kafka.common.serialization.StringDeserializer
      value-deserializer: org.apache.kafka.common.serialization.StringDeserializer
```

该片段展示属性层级，两端仅保留所需角色。讲义文章端片段显示从 `kafka:` 开始，实际应合并在 `spring.kafka` 下。配置中心保留原有属性，不要另起重复spring键覆盖ES或Mongo配置。


## 11. 搜索服务消费事件并写入ES

课程监听器 `SyncArticleListener` 消费消息，按文章ID指定文档 `_id`：

```java
@KafkaListener(topics = ArticleConstants.ARTICLE_ES_SYNC_TOPIC)
public void onMessage(String message) throws IOException {
    if (StringUtils.isBlank(message)) {
        return;
    }
    SearchArticleVo article = JSON.parseObject(message, SearchArticleVo.class);
    if (article == null || article.getId() == null) {
        throw new IllegalArgumentException("article id required");
    }
    IndexRequest request = new IndexRequest("app_info_article")
            .id(article.getId().toString())
            .source(message, XContentType.JSON);
    restHighLevelClient.index(request, RequestOptions.DEFAULT);
}
```

上方是带基本校验的整理示意。课程原代码捕获IOException、打印日志后正常返回，这可能使监听容器认为处理成功并继续提交进度，ES写入失败的消息不一定会重试。应该把失败交给明确配置的重试/死信机制，或保存可靠补偿记录。

相同文章ID重复写入覆盖同一文档，降低重复文档风险；但这不等于保证最新内容，旧事件仍可能覆盖新内容。可携带文章版本，并使用版本校验，生产消息以articleId为key固定分区。

**索引可见性：** 写入返回成功不代表立即可搜索，ES通常等待refresh后才对搜索可见。联调要区分“事件未收到”“写入失败”和“等待刷新”，不要对每条消息强制refresh造成性能下降。

事件发送、数据库更新、MinIO上传和ES写入不是一个统一事务；最终一致需要发送结果检查、重试、对账和监控。


## 12. 索引同步必须考虑上下架、删除和更新

讲义本日实现“新文章静态页生成后建立索引”。其映射没有 `isDown`/`isDelete`，搜索查询也没有对应过滤条件；第06天上下架事件只修改article配置，讲义未展示search服务同步处理上下架。

因此不能声称“文章下架后本日搜索必然立即隐藏”。生产需要补齐下面一种设计：

| 设计 | 做法 | 需要处理 |
|---|---|---|
| 文档状态字段 | 索引保存isDown/isDelete，查询filter排除 | 状态变更事件、版本与补偿 |
| 仅保存可见文章 | 下架/删除时删除文档，上架重建 | 重新上架的数据来源和旧事件 |

文章修改、标题变更也需要更新索引。初始导入、增量更新、下架事件若并发，应统一版本规则，防止旧的“创建索引”事件重新暴露已下架文章。

此外搜索列表隐藏不等于详情文件不可访问。若静态页URL仍可直接打开，详情访问也需按业务要求执行状态校验；本日讲义没有完成这条访问控制链路。

### 静态化原实现的失败边界

讲义渲染异常后打印异常，仍可能继续上传空的/不完整的HTML并发布索引事件。应在渲染、上传或URL回写失败时停止后续动作、进入可追踪的补偿。HTML转字节建议显式UTF-8，避免平台默认编码不同。


## 13. MongoDB：历史与联想词的存储模型

课程选择MongoDB保存大量用户搜索记录和公共联想词。MongoDB以集合和文档组织数据，Spring Data使用 `MongoTemplate` 查询。选择MongoDB不代表关系型数据库不能存历史，而是本课按数据特点进行技术分工。

依赖：

```xml
<dependency>
  <groupId>org.springframework.boot</groupId>
  <artifactId>spring-boot-starter-data-mongodb</artifactId>
</dependency>
```

配置示意：

```yaml
spring:
  data:
    mongodb:
      uri: ${MONGODB_URI}
      database: leadnews-history
```

课程原配置分别使用host和port（默认演示端口27017）；示例URI便于统一环境配置。没有在本次运行Docker安装或连接MongoDB。

集合：

| 集合 | 字段 | 用途 |
|---|---|---|
| ap_user_search | id、userId、keyword、createdTime | 每个用户自己的搜索历史 |
| ap_associate_words | id、associateWords、createdTime | 公共联想词库 |

实体使用 `@Document("集合名")`。Spring Data识别名为id的属性并映射Mongo `_id`，也可明确添加 `@Id`。历史ID是String，不是文章Long ID，接口不能混用。


## 14. MongoTemplate核心操作

课程 `mongo-demo` 展示保存、按ID查询、条件查询、排序和删除：

```java
// 保存文档：有匹配ID时更新，否则插入，注意它可能替换整份文档。
mongoTemplate.save(word);

// 按ID查一条。
ApAssociateWords result = mongoTemplate.findById(
        id, ApAssociateWords.class);

// 条件查询 + 时间倒序。
Query query = Query.query(Criteria.where("associateWords").is(keyword))
        .with(Sort.by(Sort.Direction.DESC, "createdTime"));
List<ApAssociateWords> words = mongoTemplate.find(
        query, ApAssociateWords.class);

// 按条件删除，执行前必须确认范围。
mongoTemplate.remove(query, ApAssociateWords.class);
```

`Criteria.is` 是精确匹配，`regex` 是正则匹配；两者语义不同。`Query` 组合筛选、排序、limit，`MongoTemplate`按实体映射操作集合。

常见补充操作：`updateFirst` 修改字段、`upsert` 不存在时插入、`findAndModify` 返回修改前后文档。`save`与`findAndReplace`是文档替换思路，修改时间这类单字段更新可用Update，避免不必要地覆盖其他字段。

索引应依据实际查询创建，例如历史常按userId筛选并按时间排序，可建 `(userId, createdTime DESC)`；去重还可建 `(userId, keyword)` 唯一索引。数据已有重复时应先清理再建唯一索引。


## 15. 搜索历史：异步保存与10条保留规则

需求：每个用户保留最新10个搜索关键词，按最近搜索时间倒序；重复搜索同一个词时刷新时间，不新增一条重复记录；超出数量时淘汰最旧记录。

```java
@Data
@Document("ap_user_search")
public class ApUserSearch {
    private String id;
    private Integer userId;
    private String keyword;
    private Date createdTime;
}
```

讲义保存流程：

```text
查询(userId, keyword)
  ├─ 已存在：刷新createdTime，save后返回
  └─ 不存在：查该用户历史，按时间倒序
        ├─ 少于10条：插入新记录
        └─ 达到10条：用新记录替换最旧记录
```

这里的 `createdTime` 被重复搜索刷新，业务语义更接近“最近搜索时间”，不要理解成不可变的首次创建时间。

课程使用 `@Async` 标记 `ApUserSearchService.insert(keyword, userId)`，启动类启用 `@EnableAsync`。搜索业务读取当前用户后显式传入userId：

```java
ApUser user = AppThreadLocalUtil.getUser();
if (user != null && dto.getFromIndex() == 0) {
    apUserSearchService.insert(dto.getSearchWords(), user.getId());
}
```

它意在仅首次搜索记录历史，后续加载更多不重复记录。搜索本身允许未登录时执行，未登录用户不保存历史；历史查看和删除要求登录。分页判断是否可靠取决于前端是否正确维护pageNum，应在接口中明确首次搜索标志或游标约定。

ThreadLocal中的用户信息不自动传递到异步线程，所以讲义直接传userId是必要的；异步线程不要再次依赖请求ThreadLocal获取身份。异步错误不应拖垮文章搜索，但需要日志与监控。


## 16. 历史保存的代码与并发边界

下面整理课程中的主要Mongo操作：

```java
Query sameWord = Query.query(Criteria.where("userId").is(userId)
        .and("keyword").is(keyword));
ApUserSearch existing = mongoTemplate.findOne(
        sameWord, ApUserSearch.class);
if (existing != null) {
    existing.setCreatedTime(new Date());
    mongoTemplate.save(existing);
    return;
}

Query allHistory = Query.query(Criteria.where("userId").is(userId))
        .with(Sort.by(Sort.Direction.DESC, "createdTime"));
List<ApUserSearch> history = mongoTemplate.find(
        allHistory, ApUserSearch.class);

// 新对象设置userId、keyword、createdTime。
if (history.size() < 10) {
    mongoTemplate.save(newHistory);
} else {
    ApUserSearch oldest = history.get(history.size() - 1);
    mongoTemplate.findAndReplace(Query.query(
            Criteria.where("id").is(oldest.getId())), newHistory);
}
```

这些是分步操作，MongoDB单次写原子不代表整个“查重 → 数量判断 → 替换”原子。并发请求可能都判断“不存在”，产生重复；也可能都看到少于10条，最终超过上限。若已有数据超过10条，替换一条也不会自动降到10条。

工程补充：

- 对关键词trim、限制长度，明确大小写/全半角等归一化规则，保证查询与唯一索引语义一致。
- 用 `(userId, keyword)` 唯一索引和upsert更新时间，处理并发DuplicateKeyException。
- “最多10条”若要求严格，需按用户协调写入，或采用一个用户一个文档、有界数组的原子更新设计；不是加唯一索引就解决数量并发。
- 查询加limit，展示层最多返回10条；保留逻辑仍要另外保证。
- 替换最旧文档要核对id映射、替换结果和用户条件；课程查询直接用旧id，补充归属条件能使意图更明确。

异步调用通常需要通过Spring代理，从其他Bean调用；同类内部直接调用不会天然触发 `@Async`。配置有界线程池及异常处理，避免搜索高峰把线程池塞满。


## 17. 查看历史：仅当前用户，时间倒序

接口：`POST /api/v1/history/load`，无请求参数。

```java
public ResponseResult findUserSearch() {
    ApUser user = AppThreadLocalUtil.getUser();
    if (user == null) {
        return ResponseResult.errorResult(AppHttpCodeEnum.NEED_LOGIN);
    }
    Query query = Query.query(Criteria.where("userId").is(user.getId()))
            .with(Sort.by(Sort.Direction.DESC, "createdTime"))
            .limit(10); // 补充：即使旧数据超限，也只展示10条。
    List<ApUserSearch> history = mongoTemplate.find(
            query, ApUserSearch.class);
    return ResponseResult.okResult(history);
}
```

Controller转交 `apUserSearchService.findUserSearch()`，路由前缀 `/api/v1/history`、方法 `/load`。课程原查询没有limit，依赖保存逻辑不超过10条；本笔记补充查询limit，防止并发或历史异常导致返回过多。

userId来自可信认证链路，不由客户端传入。搜索服务要配置与其他APP服务一致的身份解析和拦截器，在请求结束清理ThreadLocal，避免线程复用造成身份串用。

历史按最近搜索时间排序，相同时间可补充id排序让结果稳定。异步保存使刚完成搜索后立即打开历史页可能短暂看不到最新词，这属于需要明确处理的可见性延迟。


## 18. 删除历史：id与userId共同限定

接口：`POST /api/v1/history/del`，参数 `HistorySearchDto`：

```java
@Data
public class HistorySearchDto {
    private String id;
}
```

讲义按当前用户ID与记录ID共同删除，保留了正确的归属约束。下面补充dto及空白id校验：

```java
public ResponseResult delUserSearch(HistorySearchDto dto) {
    if (dto == null || StringUtils.isBlank(dto.getId())) {
        return ResponseResult.errorResult(AppHttpCodeEnum.PARAM_INVALID);
    }
    ApUser user = AppThreadLocalUtil.getUser();
    if (user == null) {
        return ResponseResult.errorResult(AppHttpCodeEnum.NEED_LOGIN);
    }
    Query query = Query.query(Criteria.where("userId").is(user.getId())
            .and("id").is(dto.getId()));
    mongoTemplate.remove(query, ApUserSearch.class);
    return ResponseResult.okResult(AppHttpCodeEnum.SUCCESS);
}
```

不能仅用id删除，也不能相信客户端提交的userId。删除不存在或不属于该用户的记录，可以按幂等语义返回成功，或者使用统一“不存在”响应，避免向请求方透露他人记录。

课程没有展示“一键清空”接口，本日实现的是单条删除。删除与仍在运行的异步新增可能交叉：删除后同一个词被新搜索重新加入属于正常业务；若要求立即清空且不被旧任务恢复，需要版本/时间边界设计。


## 19. 联想词数据来源与集合

联想词是用户输入部分关键词后显示的候选词，不等同于这个用户的历史。课程列出两类来源：分析自己业务中的高频搜索词；从外部关键词工具获取适用词库。

```java
@Data
@Document("ap_associate_words")
public class ApAssociateWords {
    private String id;
    private String associateWords;
    private Date createdTime;
}
```

讲义要求导入资料中的 `ap_associate_words.js` 到MongoDB。本篇没有读取或执行该脚本，因此不推断词库具体条数或内容。

本日只是用已有词库查询，并没有实现“每次用户搜索自动更新公共词库”或词频统计排名。生产可对词条维护热度、类别、更新时间和可用状态；需要通过去重、审核、过期策略保持建议质量。

公共联想词可能给大量用户展示，不能直接把任意用户的私人搜索记录发布到公共词库；汇总词频时应按产品约定处理隐私与质量。


## 20. 联想词接口与正则查询修正

接口：`POST /api/v1/associate/search`，复用 `UserSearchDto.searchWords/pageSize`。

讲义验证关键词非空，限制最大返回20条，再对 `associateWords` 执行包含式regex。原拼接中额外加入反斜杠，不是对整个用户输入的可靠转义；特殊字符可能改变正则含义，甚至形成非法模式。

下面是**将输入当作字面文本匹配的修正示意**：

```java
public ResponseResult findAssociate(UserSearchDto dto) {
    if (dto == null || StringUtils.isBlank(dto.getSearchWords())) {
        return ResponseResult.errorResult(AppHttpCodeEnum.PARAM_INVALID);
    }
    String keyword = dto.getSearchWords().trim();
    int size = dto.getPageSize() <= 0 ? 10 : Math.min(dto.getPageSize(), 20);

    Query query = Query.query(Criteria.where("associateWords")
            .regex(Pattern.quote(keyword)))
            .limit(size);
    List<ApAssociateWords> words = mongoTemplate.find(
            query, ApAssociateWords.class);
    return ResponseResult.okResult(words);
}
```

Mongo正则默认可匹配字符串中的片段，因此这里不需额外包 `.*`；若产品要求前缀联想，可用锚定前缀等相应策略。`Pattern.quote` 将关键词视为字面文本，要在目标Mongo/驱动版本验证边界字符的行为。

联想词需要统一默认数量、长度限制和排序。课程未设置排序，因此返回顺序不能解释为“热度最高”。大词库的任意包含式正则容易扫描大量数据，普通索引未必有效；可以按需求使用前缀索引、专门的联想索引/搜索方案和缓存。

前端补充防抖、最小输入长度及忽略过期响应，避免每敲一个字就发请求，以及旧请求结果覆盖新输入。


## 21. 原讲义实现问题汇总

| 原实现/边界 | 影响 | 笔记中的修正或建议 |
|---|---|---|
| minBehotTime未判空 | 首次搜索可能空指针 | 默认时间或明确必填 |
| pageSize未统一校验 | 0条、过量请求或参数异常 | 入口设置默认值与上限 |
| pageNum字段但ES始终from(0) | 容易误写成传统页码分页 | 说明实际时间游标；统一接口协议 |
| 只按严格小于时间分页 | 同时间文章可能漏掉 | 时间+ID稳定游标/search_after |
| query_string直接接输入 | 查询语法可能影响行为 | 普通输入使用受限查询，处理解析异常 |
| 高亮Map非空就访问title | 正文命中时可能无title高亮 | 检查具体字段并回退原title |
| 静态化异常后继续上传/同步 | 产生无效详情及索引 | 中止后续动作并补偿 |
| 索引监听器捕获失败后返回 | 消费可能成功提交而索引缺失 | 明确异常、重试与死信机制 |
| 没有搜索上下架状态同步 | 搜索可能继续展示下架文章 | 同步状态或删除索引，补充过滤 |
| 查重、限10、替换分开执行 | 并发产生重复和超限 | 唯一索引+upsert；另行保证用户配额 |
| 联想词拼接正则 | 特殊字符改变语义或解析失败 | 字面转义、limit和查询策略 |

这些说明是对讲义示例的静态分析和工程补充，未通过运行课程源码验证。本篇保留课程主线，同时明确哪些可靠性、权限和性能设计需要后续补齐。


## 22. 验证清单与排查顺序

下表是后续环境联调的验证计划，本次没有执行：

| 场景 | 预期与观察点 |
|---|---|
| IK测试 | 分词器加载成功，返回预期中文词项 |
| 初始批量导入 | 文档数和目标文章一致；Bulk失败项可追踪 |
| 标题命中 | 返回列表，h_title具有受控高亮 |
| 仅正文命中 | 返回文章，h_title安全回退原标题 |
| 空关键词/空时间 | 返回校验错误或按明确默认值处理，不发生空指针 |
| 加载更早结果 | 不反复返回上一批；相同时间边界有专门测试 |
| 新文章发布 | 静态URL有效，Kafka事件写入ES，refresh后可搜索 |
| ES写入暂时失败 | 消息重试/死信/补偿可追踪，不静默确认 |
| 文章下架/删除 | 验证是否已有索引可见性同步；讲义主线不足以保证 |
| 登录搜索 | 首次搜索异步保存；加载更多不重复保存 |
| 重复搜索同词 | 不增加重复历史，最近时间刷新 |
| 第11个关键词 | 最旧历史淘汰，并发下仍按设计约束 |
| 游客历史接口 | 返回需要登录，不能查看他人记录 |
| 删除他人记录ID | 他人数据不被删除 |
| 联想词特殊字符 | 输入当作字面文本，无正则语法异常 |
| 联想结果数量 | 默认数量有效，上限20，顺序符合业务约定 |

排查顺序：

1. **请求到不了：** 核对网关 `/search/**`、StripPrefix和服务注册名。
2. **ES搜索为空：** 核对索引名、文档内容、IK映射、关键词和时间边界。
3. **新文不可搜索：** 先查静态化成功与URL回写，再查Kafka发送、search监听、ES写入、refresh。
4. **高亮异常：** 检查title实际是否有片段、字段类型和前端渲染规则。
5. **历史不保存：** 检查登录身份、首次搜索判断、异步代理、Mongo配置及写入错误。
6. **联想为空/太慢：** 检查是否导入词库、数量默认值、字面匹配策略、词库规模和索引。

可观测指标包括搜索延迟、ES索引失败率、Kafka消费积压、历史写入失败和联想查询耗时。日志避免完整打印用户搜索历史或完整文章正文。


## 23. 复习与自测

### 必须掌握的主线

- 搜索服务查询ES而不是每次扫描文章主库；ES文档使用文章ID作为 `_id`。
- 初始索引用Bulk导入，新文静态化后通过Kafka建立增量索引。
- 布尔查询组合关键词与时间范围，按发布时间倒序；本课实际是时间分页。
- 标题高亮附加为h_title，标题未命中时回退原值。
- MongoDB历史按用户隔离：重复词刷新时间，最多10条；异步保存不阻塞搜索。
- 历史删除同时匹配userId和记录id，避免越权。
- 联想词来自公共词库，课程用regex查询；不要把它等同于私人历史或自动词频排名。
- Kafka和ES同步是最终一致，失败重试、版本控制和上下架状态同步要明确设计。

### 自测题与参考答案

1. **为什么images和staticUrl设置index:false还能返回？** 字段仍保留在 `_source`，只是不用它们建立通常的搜索索引。
2. **title和content为什么使用text而不是keyword？** 它们需要分词全文匹配；keyword适合完整值精确匹配等场景。
3. **本课的pageNum是否控制ES的from？** 没有，ES查询固定from(0)，pageNum在历史首次记录判断中使用。
4. **时间严格小于上一批最小时间有什么问题？** 同时刻剩余文档可能被跳过，需要稳定的时间+ID游标。
5. **有高亮Map就一定能获取title片段吗？** 不一定，可能只有其他字段，必须检查title并回退。
6. **同步消息重复是否必然产生重复ES文档？** 使用相同 `_id` 通常覆盖同文档，但旧事件可能覆盖新版本。
7. **文章下架后搜索一定隐藏吗？** 本日代码没有展示索引状态同步和过滤，不能保证，需补充事件链路。
8. **为什么异步保存要显式传userId？** 请求ThreadLocal不会自动传播到工作线程，且身份应来自可信认证链路。
9. **唯一索引是否保证每用户最多10条？** 不能，它解决同关键词重复；数量限制要另做并发控制。
10. **搜索词包含“+”或“(”时联想怎么处理？** 按字面文本转义，避免其被解释为正则语法，并限制查询量。

建议练习：画出“新文章从MySQL到ES”的完整链路；设计“同一时间15篇文章”的分页用例；补出下架事件与搜索过滤；说明历史保存多请求并发时如何保持去重和数量上限。练习为本笔记补充。
